# 파싱 데이터 품질 EDA

이 노트북은 \`data_parsing\`에 저장된 **파싱 결과**를 수정하지 않고 읽기 전용으로 검사합니다.

분석 범위는 다음 다섯 가지입니다.

1. 파일·레코드·컬럼 현황
2. 결측값, 저장 타입, 문자열 기반 추론 타입, 날짜 형식
3. 키 중복 및 I1250–C002 조인 가능성
4. PDF 파싱 결과의 빈 페이지·텍스트 추출률
5. 파일명과 내부 메타데이터 일치 여부

분석 결과는 \`data/eda_outputs\`에 CSV와 JSON으로 저장합니다.

## 0. 실행 원칙

- 원본 및 파싱 결과를 수정하거나 삭제하지 않습니다.
- 대용량 JSON은 전체를 한 번에 메모리에 올리지 않고 순차적으로 읽습니다.
- 파일·레코드·키·조인은 전체 데이터를 검사합니다.
- 값 분포와 타입 추론은 기본적으로 파일당 최대 10만 행을 사용합니다.
- \`FULL_SCAN=True\`로 바꾸면 값 분포와 타입도 전수 검사합니다.
- 식품원료는 \`food_raw_material.jsonl\`, \`food_raw_material_metadata.json\`만 사용합니다. 파일이 아직 없으면 다른 파일로 대체하지 않고 경고를 기록합니다.

## 1. 경로와 실행 설정

프로젝트 루트를 자동으로 찾고 보고서 저장 폴더를 준비합니다.

\`FULL_SCAN=False\`는 키와 레코드 수는 전체 검사하되, 컬럼 값의 타입·고유값·날짜 형식은 최대 \`PROFILE_SAMPLE_ROWS\`건까지만 분석하는 혼합 모드입니다.

In [1]:
from pathlib import Path

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data_parsing").is_dir():
            return candidate
    raise FileNotFoundError("data_parsing 폴더가 있는 프로젝트 루트를 찾지 못했습니다.")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_ROOT = PROJECT_ROOT / "data_parsing"
OUTPUT_DIR = PROJECT_ROOT / "data" / "eda_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FULL_SCAN = False
PROFILE_SAMPLE_ROWS = None if FULL_SCAN else 100_000
DISTINCT_SAMPLE_LIMIT = 100_000
DUPLICATE_SAMPLE_LIMIT = 100
COMPUTE_FILE_HASH = True

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"분석 대상: {DATA_ROOT}")
print(f"보고서 저장: {OUTPUT_DIR}")
print(f"실행 모드: {'전체 정밀 검사' if FULL_SCAN else '혼합 검사'}")

프로젝트 루트: C:\Users\Playdata\Desktop\mle-01-final-team3
분석 대상: C:\Users\Playdata\Desktop\mle-01-final-team3\data_parsing
보고서 저장: C:\Users\Playdata\Desktop\mle-01-final-team3\data\eda_outputs
실행 모드: 혼합 검사


## 2. 이미 확인된 문제 기록

아래 문제는 새로 탐지해야 하는 문제가 아니라 팀에서 이미 확인하여 보완 중인 항목입니다. EDA 결과에서 반복 발견되더라도 새 오류로 오해하지 않도록 별도 JSON 보고서로 저장합니다.

In [2]:
KNOWN_ISSUES = [
    {
        "issue_id": "KNOWN-001",
        "category": "duplicate_file",
        "description": "건강기능식품에 관한 법률 시행규칙과 시행령 파싱 파일의 내용이 동일함",
        "status": "파싱 담당자 보완 중",
    },
    {
        "issue_id": "KNOWN-002",
        "category": "duplicate_file",
        "description": "친환경농어업 육성 및 유기식품 등의 관리ㆍ지원에 관한 법률과 농림축산식품부 소관 시행규칙의 내용이 동일함",
        "status": "파싱 담당자 보완 중",
    },
    {
        "issue_id": "KNOWN-003",
        "category": "parser_mismatch",
        "description": "제2025-79호 HWPX 원본에 PDF 전용 파서가 적용되어 제2025-61호 PDF 결과가 잘못 저장된 것으로 추정됨",
        "status": "파싱 담당자 보완 중",
    },
    {
        "issue_id": "KNOWN-004",
        "category": "source_duplicate",
        "description": "건강기능식품 영양DB와 품목분류 CSV가 같은 내용이어서 새 데이터로 재수집함",
        "status": "파싱 담당자 보완 중",
    },
    {
        "issue_id": "KNOWN-005",
        "category": "metadata_error",
        "description": "CSV 파싱 코드의 고정 제목 때문에 C002와 I1250 내부 제목이 잘못 저장됨",
        "status": "파싱 담당자 보완 중",
    },
    {
        "issue_id": "KNOWN-006",
        "category": "file_selection",
        "description": "식품원료는 food_raw_material.jsonl과 food_raw_material_metadata.json만 사용해야 함",
        "status": "EDA 선택 규칙에 반영",
    },
]

KNOWN_ISSUES

[{'issue_id': 'KNOWN-001',
  'category': 'duplicate_file',
  'description': '건강기능식품에 관한 법률 시행규칙과 시행령 파싱 파일의 내용이 동일함',
  'status': '파싱 담당자 보완 중'},
 {'issue_id': 'KNOWN-002',
  'category': 'duplicate_file',
  'description': '친환경농어업 육성 및 유기식품 등의 관리ㆍ지원에 관한 법률과 농림축산식품부 소관 시행규칙의 내용이 동일함',
  'status': '파싱 담당자 보완 중'},
 {'issue_id': 'KNOWN-003',
  'category': 'parser_mismatch',
  'description': '제2025-79호 HWPX 원본에 PDF 전용 파서가 적용되어 제2025-61호 PDF 결과가 잘못 저장된 것으로 추정됨',
  'status': '파싱 담당자 보완 중'},
 {'issue_id': 'KNOWN-004',
  'category': 'source_duplicate',
  'description': '건강기능식품 영양DB와 품목분류 CSV가 같은 내용이어서 새 데이터로 재수집함',
  'status': '파싱 담당자 보완 중'},
 {'issue_id': 'KNOWN-005',
  'category': 'metadata_error',
  'description': 'CSV 파싱 코드의 고정 제목 때문에 C002와 I1250 내부 제목이 잘못 저장됨',
  'status': '파싱 담당자 보완 중'},
 {'issue_id': 'KNOWN-006',
  'category': 'file_selection',
  'description': '식품원료는 food_raw_material.jsonl과 food_raw_material_metadata.json만 사용해야 함',
  'status': 'EDA 선택 규칙에 반영'}]

## 3. 공통 분석 함수

다음 셀에는 대용량 JSON/JSONL 스트리밍, 결측값 판정, 날짜 형식 추론, 컬럼 프로파일, 키 프로파일, 조인 검증, PDF 추출률, 메타데이터 비교 함수가 들어 있습니다.

결측값은 \`null\`, 빈 문자열, \`-\`, \`null\`, \`none\`, \`nan\` 문자열을 포함합니다. 날짜는 값 자체가 유효한 날짜인지까지 확인합니다.

In [3]:
from __future__ import annotations

from collections import Counter, defaultdict
from datetime import datetime
from difflib import SequenceMatcher
from pathlib import Path
from statistics import mean, median
from typing import Any, Iterable, Iterator
from urllib.parse import unquote_plus
import hashlib
import json
import math
import re
import unicodedata

import pandas as pd


MISSING_STRINGS = {"", "-", "null", "none", "nan", "na", "n/a"}
PREFERRED_RAW_MATERIAL_FILES = {
    "food_raw_material.jsonl",
    "food_raw_material_metadata.json",
}
KEY_CANDIDATES = (
    "PRDLST_REPORT_NO",
    "LCNS_NO",
    "DSPSDTLS_SEQ",
    "HF_FNCLTY_MTRAL_RCOGN_NO",
)
DATE_NAME_PATTERN = re.compile(
    r"(^|_)(DT|DATE|DATETIME|DTTM|BGNDT|ENDDT|PRMS_DT|CHNG_DT|PUBLIC_DT|LAST_UPDT_DTM)($|_)"
    r"|(_AT$)|날짜|일자|시행일|공포일",
    re.IGNORECASE,
)


def is_missing(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    return isinstance(value, str) and value.strip().lower() in MISSING_STRINGS


def detect_date_format(value: Any) -> str | None:
    if not isinstance(value, str):
        return None
    text = value.strip()
    formats = (
        (r"\d{8}", "%Y%m%d", "YYYYMMDD"),
        (r"\d{4}-\d{2}-\d{2}", "%Y-%m-%d", "YYYY-MM-DD"),
        (r"\d{4}\.\d{2}\.\d{2}", "%Y.%m.%d", "YYYY.MM.DD"),
        (r"\d{4}/\d{2}/\d{2}", "%Y/%m/%d", "YYYY/MM/DD"),
    )
    for pattern, parser_format, label in formats:
        if re.fullmatch(pattern, text):
            try:
                datetime.strptime(text, parser_format)
                return label
            except ValueError:
                return None
    if re.fullmatch(
        r"\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}:\d{2}(?:\.\d+)?(?:Z|[+-]\d{2}:\d{2})?",
        text,
    ):
        try:
            datetime.fromisoformat(text.replace("Z", "+00:00"))
            return "ISO_DATETIME"
        except ValueError:
            return None
    return None


def is_date_column(column_name: str) -> bool:
    return bool(DATE_NAME_PATTERN.search(str(column_name)))


def infer_scalar_type(value: Any) -> str:
    if is_missing(value):
        return "missing"
    if isinstance(value, bool):
        return "boolean"
    if isinstance(value, int):
        return "integer"
    if isinstance(value, float):
        return "float"
    if isinstance(value, list):
        return "array"
    if isinstance(value, dict):
        return "object"
    if isinstance(value, str):
        text = value.strip()
        if text.lower() in {"true", "false", "y", "n", "yes", "no", "예", "아니오", "o", "x"}:
            return "boolean_string"
        if re.fullmatch(r"[+-]?\d+", text):
            return "date_string" if detect_date_format(text) else "integer_string"
        if re.fullmatch(r"[+-]?(?:\d+\.\d*|\d*\.\d+)", text):
            return "float_string"
        if detect_date_format(text):
            return "date_string"
        return "string"
    return type(value).__name__


def iter_json_array(
    path: str | Path,
    field: str | None = None,
    chunk_size: int = 1024 * 1024,
) -> Iterator[Any]:
    path = Path(path)
    decoder = json.JSONDecoder()
    with path.open("r", encoding="utf-8-sig") as stream:
        buffer = ""
        position = 0
        eof = False

        if field is None:
            while True:
                chunk = stream.read(chunk_size)
                if not chunk:
                    raise ValueError(f"JSON 배열을 찾지 못했습니다: {path}")
                buffer += chunk
                match = re.search(r"\[", buffer)
                if match:
                    position = match.end()
                    break
        else:
            pattern = re.compile(rf'"{re.escape(field)}"\s*:\s*\[')
            while True:
                chunk = stream.read(chunk_size)
                if not chunk:
                    raise ValueError(f"'{field}' 배열을 찾지 못했습니다: {path}")
                buffer += chunk
                match = pattern.search(buffer)
                if match:
                    position = match.end()
                    break
                if len(buffer) > 8192:
                    buffer = buffer[-8192:]

        while True:
            while True:
                while position < len(buffer) and (
                    buffer[position].isspace() or buffer[position] == ","
                ):
                    position += 1
                if position < len(buffer):
                    break
                if eof:
                    raise ValueError(f"JSON 배열이 닫히지 않았습니다: {path}")
                buffer = ""
                position = 0
                chunk = stream.read(chunk_size)
                if chunk:
                    buffer = chunk
                else:
                    eof = True

            if buffer[position] == "]":
                return

            while True:
                try:
                    value, end = decoder.raw_decode(buffer, position)
                    break
                except json.JSONDecodeError as error:
                    chunk = stream.read(chunk_size)
                    if not chunk:
                        raise ValueError(
                            f"JSON 레코드 해석 실패: {path}, 위치 {error.pos}"
                        ) from error
                    if position:
                        buffer = buffer[position:] + chunk
                        position = 0
                    else:
                        buffer += chunk

            yield value
            position = end
            if position > chunk_size:
                buffer = buffer[position:]
                position = 0


def iter_jsonl(path: str | Path) -> Iterator[dict[str, Any]]:
    with Path(path).open("r", encoding="utf-8-sig") as stream:
        for line_number, line in enumerate(stream, start=1):
            text = line.strip()
            if not text:
                continue
            value = json.loads(text)
            if not isinstance(value, dict):
                raise ValueError(f"JSONL {line_number}행이 객체가 아닙니다: {path}")
            yield value


def read_prefix(path: str | Path, char_count: int = 131_072) -> str:
    with Path(path).open("r", encoding="utf-8-sig") as stream:
        return stream.read(char_count)


def detect_json_shape(path: str | Path) -> str:
    path = Path(path)
    if path.suffix.lower() == ".jsonl":
        return "jsonl"
    prefix = read_prefix(path)
    stripped = prefix.lstrip()
    if stripped.startswith("["):
        return "root_array"
    if re.search(r'"records"\s*:\s*\[', prefix):
        return "records"
    if re.search(r'"pages"\s*:\s*\[', prefix):
        return "pages"
    return "object"


def iter_dataset_records(
    path: str | Path,
    shape: str | None = None,
) -> Iterator[dict[str, Any]]:
    path = Path(path)
    shape = shape or detect_json_shape(path)
    if shape == "jsonl":
        yield from iter_jsonl(path)
    elif shape == "root_array":
        yield from iter_json_array(path)
    elif shape in {"records", "pages"}:
        yield from iter_json_array(path, shape)
    elif shape == "object":
        with path.open("r", encoding="utf-8-sig") as stream:
            value = json.load(stream)
        if isinstance(value, dict):
            yield value
        else:
            raise ValueError(f"객체형 JSON이 아닙니다: {path}")
    else:
        raise ValueError(f"지원하지 않는 JSON 구조입니다: {shape}")


def discover_files(root: str | Path) -> list[dict[str, Any]]:
    root = Path(root)
    rows = []
    for path in sorted(root.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in {".json", ".jsonl"}:
            continue
        relative = path.relative_to(root)
        in_raw_material_dir = "식품원료" in relative.parts
        included = True
        exclusion_reason = ""
        if in_raw_material_dir and path.name not in PREFERRED_RAW_MATERIAL_FILES:
            included = False
            exclusion_reason = "식품원료 지정 파일이 아니므로 제외"
        try:
            shape = detect_json_shape(path)
            shape_error = ""
        except Exception as error:
            shape = "unreadable"
            shape_error = f"{type(error).__name__}: {error}"
        rows.append(
            {
                "relative_path": relative.as_posix(),
                "file_name": path.name,
                "extension": path.suffix.lower(),
                "size_bytes": path.stat().st_size,
                "size_mb": round(path.stat().st_size / (1024**2), 3),
                "included": included,
                "exclusion_reason": exclusion_reason,
                "json_shape": shape,
                "shape_error": shape_error,
            }
        )
    return rows


def sha256_file(path: str | Path, chunk_size: int = 4 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def extract_declared_metadata(path: str | Path) -> dict[str, Any]:
    prefix = read_prefix(path)
    result = {}
    patterns = {
        "internal_title": r'"title"\s*:\s*"([^"]*)"',
        "source_path": r'"source_(?:pdf|csv)"\s*:\s*"([^"]*)"',
        "declared_record_count": r'"record_count"\s*:\s*(\d+)',
        "declared_page_count": r'"page_count"\s*:\s*(\d+)',
        "notice_number": r'"notice_number"\s*:\s*"([^"]*)"',
        "published_date": r'"published_date"\s*:\s*"([^"]*)"',
    }
    for key, pattern in patterns.items():
        match = re.search(pattern, prefix)
        result[key] = match.group(1) if match else None
    for key in ("declared_record_count", "declared_page_count"):
        if result[key] is not None:
            result[key] = int(result[key])
    return result


def _empty_column_state() -> dict[str, Any]:
    return {
        "present_count": 0,
        "non_missing_count": 0,
        "sampled_non_missing_count": 0,
        "physical_types": Counter(),
        "inferred_types": Counter(),
        "distinct_values": set(),
        "distinct_truncated": False,
        "min_length": None,
        "max_length": None,
        "date_formats": Counter(),
        "date_invalid_count": 0,
    }


def profile_dataset(
    path: str | Path,
    shape: str | None = None,
    profile_limit: int | None = 100_000,
    distinct_limit: int = 100_000,
    key_candidates: Iterable[str] = KEY_CANDIDATES,
) -> dict[str, Any]:
    path = Path(path)
    shape = shape or detect_json_shape(path)
    columns: dict[str, dict[str, Any]] = defaultdict(_empty_column_state)
    key_counters = {key: Counter() for key in key_candidates}
    row_count = 0
    sampled_row_count = 0
    license_by_product: dict[str, str | None] = {}

    for record in iter_dataset_records(path, shape):
        row_count += 1
        if not isinstance(record, dict):
            record = {"value": record}
        should_profile = profile_limit is None or sampled_row_count < profile_limit
        if should_profile:
            sampled_row_count += 1

        for column, value in record.items():
            state = columns[str(column)]
            state["present_count"] += 1
            missing = is_missing(value)
            if not missing:
                state["non_missing_count"] += 1
            if not should_profile or missing:
                continue
            state["sampled_non_missing_count"] += 1
            state["physical_types"][type(value).__name__] += 1
            inferred = infer_scalar_type(value)
            state["inferred_types"][inferred] += 1
            if not state["distinct_truncated"]:
                try:
                    state["distinct_values"].add(
                        json.dumps(value, ensure_ascii=False, sort_keys=True)
                        if isinstance(value, (dict, list))
                        else str(value)
                    )
                except TypeError:
                    state["distinct_values"].add(repr(value))
                if len(state["distinct_values"]) > distinct_limit:
                    state["distinct_truncated"] = True
                    state["distinct_values"].clear()
            length = len(str(value))
            state["min_length"] = length if state["min_length"] is None else min(state["min_length"], length)
            state["max_length"] = length if state["max_length"] is None else max(state["max_length"], length)
            if is_date_column(str(column)):
                date_format = detect_date_format(value)
                if date_format:
                    state["date_formats"][date_format] += 1
                else:
                    state["date_invalid_count"] += 1

        for key, counter in key_counters.items():
            value = record.get(key)
            if not is_missing(value):
                counter[str(value).strip()] += 1

        product_id = record.get("PRDLST_REPORT_NO")
        if not is_missing(product_id):
            product_id = str(product_id).strip()
            license_no = record.get("LCNS_NO")
            license_no = None if is_missing(license_no) else str(license_no).strip()
            previous = license_by_product.get(product_id)
            if product_id not in license_by_product:
                license_by_product[product_id] = license_no
            elif previous != license_no:
                license_by_product[product_id] = "__MULTIPLE__"

    metadata = extract_declared_metadata(path)
    declared_count = (
        metadata["declared_page_count"]
        if shape == "pages"
        else metadata["declared_record_count"]
        if shape in {"records", "root_array", "jsonl"}
        else None
    )
    dataset_row = {
        "file_name": path.name,
        "json_shape": shape,
        "actual_record_or_page_count": row_count,
        "declared_count": declared_count,
        "declared_count_matches": None if declared_count is None else declared_count == row_count,
        "column_count": len(columns),
        "profiled_row_count": sampled_row_count,
        "profile_mode": "full" if profile_limit is None or sampled_row_count == row_count else "sampled_values_full_keys",
        "error": "",
    }

    column_rows = []
    date_rows = []
    for column, state in sorted(columns.items()):
        missing_count = row_count - state["non_missing_count"]
        column_rows.append(
            {
                "file_name": path.name,
                "column_name": column,
                "row_count": row_count,
                "present_count": state["present_count"],
                "non_missing_count": state["non_missing_count"],
                "missing_count": missing_count,
                "missing_rate": missing_count / row_count if row_count else None,
                "profiled_non_missing_count": state["sampled_non_missing_count"],
                "physical_types": json.dumps(state["physical_types"], ensure_ascii=False),
                "inferred_types": json.dumps(state["inferred_types"], ensure_ascii=False),
                "distinct_sample_count": None if state["distinct_truncated"] else len(state["distinct_values"]),
                "distinct_truncated": state["distinct_truncated"],
                "min_string_length": state["min_length"],
                "max_string_length": state["max_length"],
            }
        )
        if is_date_column(column):
            valid_count = sum(state["date_formats"].values())
            date_rows.append(
                {
                    "file_name": path.name,
                    "column_name": column,
                    "sampled_non_missing_count": state["sampled_non_missing_count"],
                    "valid_date_count": valid_count,
                    "invalid_date_count": state["date_invalid_count"],
                    "valid_date_rate": valid_count / state["sampled_non_missing_count"]
                    if state["sampled_non_missing_count"]
                    else None,
                    "detected_formats": json.dumps(state["date_formats"], ensure_ascii=False),
                }
            )

    key_rows = []
    duplicate_samples = []
    retained_counters = {}
    for key, counter in key_counters.items():
        if not counter:
            continue
        retained_counters[key] = counter
        duplicate_pairs = [(value, count) for value, count in counter.items() if count > 1]
        duplicate_pairs.sort(key=lambda item: (-item[1], item[0]))
        key_rows.append(
            {
                "file_name": path.name,
                "key_column": key,
                "row_count": row_count,
                "non_missing_key_count": sum(counter.values()),
                "missing_key_count": row_count - sum(counter.values()),
                "unique_key_count": len(counter),
                "duplicate_key_count": len(duplicate_pairs),
                "duplicate_row_count": sum(count - 1 for _, count in duplicate_pairs),
                "max_rows_per_key": max(counter.values()),
                "is_unique": len(counter) == sum(counter.values()) == row_count,
            }
        )
        for value, count in duplicate_pairs[:100]:
            duplicate_samples.append(
                {
                    "file_name": path.name,
                    "key_column": key,
                    "key_value": value,
                    "row_count_for_key": count,
                }
            )

    return {
        "dataset_row": dataset_row,
        "column_rows": column_rows,
        "date_rows": date_rows,
        "key_rows": key_rows,
        "duplicate_samples": duplicate_samples,
        "key_counters": retained_counters,
        "license_by_product": license_by_product,
    }


def profile_pdf_pages(path: str | Path) -> dict[str, Any]:
    path = Path(path)
    metadata = extract_declared_metadata(path)
    character_counts = []
    empty_count = 0
    replacement_count = 0
    for page in iter_json_array(path, "pages"):
        text = page.get("text") if isinstance(page, dict) else None
        text = "" if text is None else str(text)
        count = len(text.strip())
        character_counts.append(count)
        empty_count += int(count == 0)
        replacement_count += text.count("�")
    actual_count = len(character_counts)
    return {
        "file_name": path.name,
        "declared_page_count": metadata["declared_page_count"],
        "actual_page_count": actual_count,
        "page_count_matches": metadata["declared_page_count"] == actual_count
        if metadata["declared_page_count"] is not None
        else None,
        "empty_page_count": empty_count,
        "nonempty_page_count": actual_count - empty_count,
        "nonempty_page_rate": (actual_count - empty_count) / actual_count if actual_count else None,
        "total_extracted_characters": sum(character_counts),
        "average_characters_per_page": mean(character_counts) if character_counts else None,
        "median_characters_per_page": median(character_counts) if character_counts else None,
        "replacement_character_count": replacement_count,
        "needs_reparse_review": bool(
            actual_count
            and (
                empty_count / actual_count >= 0.2
                or sum(character_counts) / actual_count < 100
                or replacement_count > 0
            )
        ),
    }


def profile_join(
    left_counter: dict[str, int] | Counter,
    right_counter: dict[str, int] | Counter,
    left_license: dict[str, str | None] | None = None,
    right_license: dict[str, str | None] | None = None,
) -> dict[str, Any]:
    left_counter = Counter(left_counter)
    right_counter = Counter(right_counter)
    left_keys = set(left_counter)
    right_keys = set(right_counter)
    both = left_keys & right_keys
    left_license = left_license or {}
    right_license = right_license or {}
    license_match = 0
    license_mismatch = 0
    license_missing = 0
    for key in both:
        left_value = left_license.get(key)
        right_value = right_license.get(key)
        if left_value in {None, "__MULTIPLE__"} or right_value in {None, "__MULTIPLE__"}:
            license_missing += 1
        elif left_value == right_value:
            license_match += 1
        else:
            license_mismatch += 1
    estimated_rows = sum(left_counter[key] * right_counter[key] for key in both)
    return {
        "join_key": "PRDLST_REPORT_NO",
        "left_unique_key_count": len(left_keys),
        "right_unique_key_count": len(right_keys),
        "both_key_count": len(both),
        "left_only_key_count": len(left_keys - right_keys),
        "right_only_key_count": len(right_keys - left_keys),
        "left_overlap_rate": len(both) / len(left_keys) if left_keys else None,
        "right_overlap_rate": len(both) / len(right_keys) if right_keys else None,
        "left_duplicate_key_count": sum(count > 1 for count in left_counter.values()),
        "right_duplicate_key_count": sum(count > 1 for count in right_counter.values()),
        "estimated_inner_join_rows": estimated_rows,
        "join_row_inflation_over_unique_matches": estimated_rows - len(both),
        "one_to_one_safe": all(left_counter[key] == right_counter[key] == 1 for key in both),
        "license_match_count": license_match,
        "license_mismatch_count": license_mismatch,
        "license_missing_or_conflicting_count": license_missing,
    }


def normalize_name(value: str | None) -> str:
    if not value:
        return ""
    text = unquote_plus(str(value))
    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"\.(jsonl?|csv|pdf|hwpx?)$", "", text)
    return re.sub(r"[^0-9a-z가-힣]+", "", text)


def recursive_find_title(value: Any) -> str | None:
    title_keys = ("법령명_한글", "행정규칙명", "title")
    if isinstance(value, dict):
        for key in title_keys:
            title = value.get(key)
            if isinstance(title, str) and title.strip():
                return title.strip()
        for child in value.values():
            found = recursive_find_title(child)
            if found:
                return found
    elif isinstance(value, list):
        for child in value[:20]:
            found = recursive_find_title(child)
            if found:
                return found
    return None


def compare_metadata(path: str | Path, shape: str | None = None) -> dict[str, Any]:
    path = Path(path)
    shape = shape or detect_json_shape(path)
    metadata = extract_declared_metadata(path)
    title = metadata["internal_title"]
    if not title and shape == "object" and path.stat().st_size < 20 * 1024 * 1024:
        try:
            with path.open("r", encoding="utf-8-sig") as stream:
                title = recursive_find_title(json.load(stream))
        except Exception:
            title = None
    source_path = metadata["source_path"]
    source_name = Path(source_path.replace("\\\\", "/")).name if source_path else None
    normalized_file = normalize_name(path.stem)
    normalized_source = normalize_name(Path(source_name).stem) if source_name else ""
    normalized_title = normalize_name(title)
    source_match = normalized_file == normalized_source if source_name else None
    title_similarity = (
        SequenceMatcher(None, normalized_file, normalized_title).ratio()
        if normalized_file and normalized_title
        else None
    )
    known_issue = ""
    if "2025-79" in path.name:
        known_issue = "KNOWN-003"
    elif path.name in {"C002_261123.json", "I1250_261123.json"}:
        known_issue = "KNOWN-005"

    if not title and not source_name:
        status = "metadata_missing_or_not_applicable"
    elif source_match is False or (title_similarity is not None and title_similarity < 0.35):
        status = "mismatch"
    else:
        status = "match_or_review"
    if known_issue:
        status = "known_issue"

    return {
        "file_name": path.name,
        "internal_title": title,
        "source_name": source_name,
        "source_name_matches_file": source_match,
        "title_filename_similarity": title_similarity,
        "metadata_status": status,
        "known_issue_id": known_issue,
        "notice_number": metadata["notice_number"],
        "published_date": metadata["published_date"],
    }


## 4. 분석 함수 자체 점검

실제 1.5GB 데이터를 읽기 전에 작은 임시 데이터로 핵심 동작을 확인합니다. 이 셀의 assertion이 실패하면 본 분석을 진행하지 않는 것이 안전합니다.

In [4]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as temporary_directory:
    temp = Path(temporary_directory)
    (temp / "root.json").write_text('[{"id": 1}, {"id": 2}]', encoding="utf-8")
    (temp / "named.json").write_text(
        '{"document": {"record_count": 2}, "records": [{"id": 3}, {"id": 4}]}',
        encoding="utf-8",
    )
    (temp / "rows.jsonl").write_text('{"id": 5}\n{"id": 6}\n', encoding="utf-8")
    assert [row["id"] for row in iter_json_array(temp / "root.json")] == [1, 2]
    assert [row["id"] for row in iter_json_array(temp / "named.json", "records")] == [3, 4]
    assert [row["id"] for row in iter_jsonl(temp / "rows.jsonl")] == [5, 6]

assert is_missing(None) and is_missing("") and is_missing("-")
assert not is_missing("0")
assert detect_date_format("20261124") == "YYYYMMDD"
assert detect_date_format("2026-11-24 10:11:12.123") == "ISO_DATETIME"
assert detect_date_format("20261340") is None

print("핵심 함수 자체 점검 통과")

핵심 함수 자체 점검 통과


## 5. 파일 현황과 중복 파일 확인

모든 JSON/JSONL의 경로, 크기, 구조를 조사합니다. SHA-256 해시가 같은 파일은 내용이 완전히 동일합니다.

식품원료 폴더의 지정 파일이 아직 준비되지 않았다면 \`included=False\`와 제외 사유가 기록됩니다.

In [5]:
file_inventory_df = pd.DataFrame(discover_files(DATA_ROOT))

if COMPUTE_FILE_HASH:
    hashes = []
    for index, row in file_inventory_df.iterrows():
        path = DATA_ROOT / row["relative_path"]
        print(f"[{index + 1}/{len(file_inventory_df)}] 해시 계산: {row['relative_path']}")
        hashes.append(sha256_file(path))
    file_inventory_df["sha256"] = hashes
    hash_counts = file_inventory_df["sha256"].value_counts()
    file_inventory_df["exact_duplicate_file"] = file_inventory_df["sha256"].map(hash_counts).gt(1)
    duplicate_hashes = {
        value: f"DUP-{number:03d}"
        for number, value in enumerate(
            sorted(hash_counts[hash_counts > 1].index), start=1
        )
    }
    file_inventory_df["duplicate_group_id"] = file_inventory_df["sha256"].map(duplicate_hashes).fillna("")
else:
    file_inventory_df["sha256"] = ""
    file_inventory_df["exact_duplicate_file"] = False
    file_inventory_df["duplicate_group_id"] = ""

preferred_status = file_inventory_df[
    file_inventory_df["file_name"].isin(PREFERRED_RAW_MATERIAL_FILES)
][["file_name", "included", "relative_path"]]
if len(preferred_status) < 2:
    print(
        "경고: 지정 식품원료 파일 2개가 모두 준비되지 않았습니다. "
        "기존 타임스탬프 파일은 분석에 사용하지 않습니다."
    )

display(
    file_inventory_df.groupby(["extension", "included"], dropna=False)
    .agg(file_count=("file_name", "size"), total_mb=("size_mb", "sum"))
    .reset_index()
)
display(file_inventory_df[file_inventory_df["exact_duplicate_file"]])

[1/59] 해시 계산: 00_공통_표시광고-20261123T025100Z-1-001/과거/식품등의 부당한 표시 또는 광고의 내용 기준_제2021-89호.json
[2/59] 해시 계산: 00_공통_표시광고-20261123T025100Z-1-001/현재/★++「식품등의+부당한+표시+또는+광고의+내용+기준」+고시+전문(제2025-79호,+2025.12.4..json
[3/59] 해시 계산: 01_건강기능식품/법령_고시/원본 데이터/시행예정/2028/건강기능식품의 표시기준(식품의약품안전처고시)(제2026-66호)(20280101).json
[4/59] 해시 계산: 01_건강기능식품/법령_고시/현재/기준규격/제2026-43호/건강기능식품의 기준 및 규격(식품의약품안전처고시)(제2026-43호)(20260611).json
[5/59] 해시 계산: 01_건강기능식품/법령_고시/현재/기준규격/제2026-43호/건강기능식품의 기준 및 규격_(제1~제3)_개정.json
[6/59] 해시 계산: 01_건강기능식품/법령_고시/현재/기준규격/제2026-43호/건강기능식품의 기준 및 규격_(제4).json
[7/59] 해시 계산: 01_건강기능식품/법령_고시/현재/기준규격/제2026-43호/건강기능식품의 기준 및 규격_(제5).json
[8/59] 해시 계산: 01_건강기능식품/법령_고시/현재/표시기준/★건강기능식품의 표시기준 고시전문(고시 제2025-61호, 2025.8.29.).json
[9/59] 해시 계산: 01_건강기능식품/원본 데이터/개별인정-20261123T031709Z-1-001/건강기능식품_개별인정형_정보.json
[10/59] 해시 계산: 01_건강기능식품/원본 데이터/기능성원료-20261123T031711Z-1-001/건강기능식품_기능성_원료인정현황.json
[11/59] 해시 계산: 01_건강기능식품/원본 데이터/영양DB-20261123T031713Z-1-001/건강기능식품_영양DB.json
[12/59] 해시 계산: 01_건강기능식품/원본 데이터/제품정보-2

,extension,included,file_count,total_mb
0,.json,True,58,1506.383
1,.jsonl,True,1,4.180


,relative_path,file_name,extension,size_bytes,size_mb,included,exclusion_reason,json_shape,shape_error,sha256,exact_duplicate_file,duplicate_group_id
33,law/건강기능식품에 관한 법률 시행규칙.json,건강기능식품에 관한 법률 시행규칙.json,.json,133185,0.127,True,,object,,c371c965fd483f2284bddca0c384590b39c18f372216e0...,True,DUP-002
34,law/건강기능식품에 관한 법률 시행령.json,건강기능식품에 관한 법률 시행령.json,.json,133185,0.127,True,,object,,c371c965fd483f2284bddca0c384590b39c18f372216e0...,True,DUP-002
37,law/농림축산식품부 소관 친환경농어업 육성 및 유기식품 등의 관리ㆍ지원에 관한 법...,농림축산식품부 소관 친환경농어업 육성 및 유기식품 등의 관리ㆍ지원에 관한 법률 시행...,.json,1443227,1.376,True,,object,,38746e9c6cdecfa3b2b360e0efd9e6405d05488fde3db5...,True,DUP-001
48,law/친환경농어업 육성 및 유기식품 등의 관리ㆍ지원에 관한 법률.json,친환경농어업 육성 및 유기식품 등의 관리ㆍ지원에 관한 법률.json,.json,1443227,1.376,True,,object,,38746e9c6cdecfa3b2b360e0efd9e6405d05488fde3db5...,True,DUP-001


## 6. 레코드·컬럼·결측·타입·날짜 프로파일

각 포함 파일을 순차적으로 읽습니다.

- 레코드 수와 키 카운트는 전수 검사합니다.
- 결측 수는 전체 행 기준입니다.
- 저장 타입, 추론 타입, 고유값 수, 날짜 형식은 혼합 모드에서 최대 10만 행을 사용합니다.
- 파싱 실패 파일이 있어도 전체 작업을 중단하지 않고 오류를 \`dataset_profile\`에 기록합니다.

대용량 파일 때문에 이 셀은 실행 시간이 가장 깁니다.

In [6]:
dataset_rows = []
column_rows = []
date_rows = []
key_rows = []
duplicate_key_samples = []
key_counters_by_file = {}
license_maps_by_file = {}

included_rows = file_inventory_df[file_inventory_df["included"]].reset_index(drop=True)
for index, inventory_row in included_rows.iterrows():
    relative_path = inventory_row["relative_path"]
    path = DATA_ROOT / relative_path
    shape = inventory_row["json_shape"]
    print(f"[{index + 1}/{len(included_rows)}] 프로파일링: {relative_path}")
    try:
        result = profile_dataset(
            path,
            shape=shape,
            profile_limit=PROFILE_SAMPLE_ROWS,
            distinct_limit=DISTINCT_SAMPLE_LIMIT,
        )
        result["dataset_row"]["relative_path"] = relative_path
        dataset_rows.append(result["dataset_row"])
        for row in result["column_rows"]:
            row["relative_path"] = relative_path
            column_rows.append(row)
        for row in result["date_rows"]:
            row["relative_path"] = relative_path
            date_rows.append(row)
        for row in result["key_rows"]:
            row["relative_path"] = relative_path
            key_rows.append(row)
        for row in result["duplicate_samples"][:DUPLICATE_SAMPLE_LIMIT]:
            row["relative_path"] = relative_path
            duplicate_key_samples.append(row)
        key_counters_by_file[path.name] = result["key_counters"]
        license_maps_by_file[path.name] = result["license_by_product"]
    except Exception as error:
        dataset_rows.append(
            {
                "file_name": path.name,
                "relative_path": relative_path,
                "json_shape": shape,
                "actual_record_or_page_count": None,
                "declared_count": None,
                "declared_count_matches": None,
                "column_count": None,
                "profiled_row_count": None,
                "profile_mode": None,
                "error": f"{type(error).__name__}: {error}",
            }
        )

dataset_profile_df = pd.DataFrame(dataset_rows)
column_profile_df = pd.DataFrame(column_rows)
date_profile_df = pd.DataFrame(date_rows)
key_profile_df = pd.DataFrame(key_rows)
duplicate_key_samples_df = pd.DataFrame(duplicate_key_samples)

display(dataset_profile_df.sort_values("actual_record_or_page_count", ascending=False))
display(
    column_profile_df.sort_values(
        ["missing_rate", "row_count"], ascending=[False, False]
    ).head(50)
)
display(date_profile_df.sort_values("valid_date_rate").head(50))

[1/59] 프로파일링: 00_공통_표시광고-20261123T025100Z-1-001/과거/식품등의 부당한 표시 또는 광고의 내용 기준_제2021-89호.json
[2/59] 프로파일링: 00_공통_표시광고-20261123T025100Z-1-001/현재/★++「식품등의+부당한+표시+또는+광고의+내용+기준」+고시+전문(제2025-79호,+2025.12.4..json
[3/59] 프로파일링: 01_건강기능식품/법령_고시/원본 데이터/시행예정/2028/건강기능식품의 표시기준(식품의약품안전처고시)(제2026-66호)(20280101).json
[4/59] 프로파일링: 01_건강기능식품/법령_고시/현재/기준규격/제2026-43호/건강기능식품의 기준 및 규격(식품의약품안전처고시)(제2026-43호)(20260611).json
[5/59] 프로파일링: 01_건강기능식품/법령_고시/현재/기준규격/제2026-43호/건강기능식품의 기준 및 규격_(제1~제3)_개정.json
[6/59] 프로파일링: 01_건강기능식품/법령_고시/현재/기준규격/제2026-43호/건강기능식품의 기준 및 규격_(제4).json
[7/59] 프로파일링: 01_건강기능식품/법령_고시/현재/기준규격/제2026-43호/건강기능식품의 기준 및 규격_(제5).json
[8/59] 프로파일링: 01_건강기능식품/법령_고시/현재/표시기준/★건강기능식품의 표시기준 고시전문(고시 제2025-61호, 2025.8.29.).json
[9/59] 프로파일링: 01_건강기능식품/원본 데이터/개별인정-20261123T031709Z-1-001/건강기능식품_개별인정형_정보.json
[10/59] 프로파일링: 01_건강기능식품/원본 데이터/기능성원료-20261123T031711Z-1-001/건강기능식품_기능성_원료인정현황.json
[11/59] 프로파일링: 01_건강기능식품/원본 데이터/영양DB-20261123T031713Z-1-001/건강기능식품_영양DB.json
[12/59] 프로파일링: 01_건강기능식품/원본 데이터/제품정보-2

,file_name,json_shape,actual_record_or_page_count,declared_count,declared_count_matches,column_count,profiled_row_count,profile_mode,error,relative_path
17,C002_261123.json,records,1087964,1087964.0,True,10,100000,sampled_values_full_keys,,02_일반식품/원본 데이터/C002/C002_261123.json
18,I1250_261123.json,records,1087955,1087955.0,True,18,100000,sampled_values_full_keys,,02_일반식품/원본 데이터/I1250/I1250_261123.json
11,I0030_20261122_202624.json,records,46501,46501.0,True,25,46501,full,,01_건강기능식품/원본 데이터/제품정보-20261123T031715Z-1-001/I...
19,food_raw_material.jsonl,jsonl,13880,NaN,None,9,13880,full,,02_일반식품/원본 데이터/식품원료/food_raw_material.jsonl
57,식품관련법 위반사례_식품안전나라.json,root_array,5060,NaN,None,17,5060,full,,"해설서, 사례/식품관련법 위반사례_식품안전나라.json"
9,건강기능식품_기능성_원료인정현황.json,records,779,779.0,True,9,779,full,,01_건강기능식품/원본 데이터/기능성원료-20261123T031711Z-1-001/...
10,건강기능식품_영양DB.json,records,585,585.0,True,8,585,full,,01_건강기능식품/원본 데이터/영양DB-20261123T031713Z-1-001/건...
12,건강기능식품_품목분류정보.json,records,585,585.0,True,8,585,full,,01_건강기능식품/원본 데이터/품목분류-20261123T031718Z-1-001/건...
5,건강기능식품의 기준 및 규격_(제4).json,pages,558,558.0,True,4,558,full,,01_건강기능식품/법령_고시/현재/기준규격/제2026-43호/건강기능식품의 기준 및...
8,건강기능식품_개별인정형_정보.json,records,457,457.0,True,7,457,full,,01_건강기능식품/원본 데이터/개별인정-20261123T031709Z-1-001/건...


,file_name,column_name,row_count,present_count,non_missing_count,missing_count,missing_rate,profiled_non_missing_count,physical_types,inferred_types,distinct_sample_count,distinct_truncated,min_string_length,max_string_length,relative_path
58,I0030_20261122_202624.json,CHILD_CRTFC_YN,46501,46501,0,46501,1.000000,0,{},{},0,False,NaN,NaN,01_건강기능식품/원본 데이터/제품정보-20261123T031715Z-1-001/I...
137,food_raw_material.jsonl,RAWMTRL_STATS_CD_NM,13880,13880,0,13880,1.000000,0,{},{},0,False,NaN,NaN,02_일반식품/원본 데이터/식품원료/food_raw_material.jsonl
257,식품관련법 위반사례_식품안전나라.json,PRSDNT_NM,5060,5060,0,5060,1.000000,0,{},{},0,False,NaN,NaN,"해설서, 사례/식품관련법 위반사례_식품안전나라.json"
259,식품관련법 위반사례_식품안전나라.json,TELNO,5060,5060,0,5060,1.000000,0,{},{},0,False,NaN,NaN,"해설서, 사례/식품관련법 위반사례_식품안전나라.json"
116,I1250_261123.json,CHILD_CRTFC_YN,1087955,1087955,442,1087513,0.999594,26,"{""str"": 26}","{""boolean_string"": 26}",1,False,1.0,1.0,02_일반식품/원본 데이터/I1250/I1250_261123.json
130,I1250_261123.json,PRPOS,1087955,1087955,521,1087434,0.999521,0,{},{},0,False,NaN,NaN,02_일반식품/원본 데이터/I1250/I1250_261123.json
226,20220406_온라인_식품·건강기능식품·의약외품·화장품_분야_부당광고_사례집-웹게...,text,84,84,1,83,0.988095,1,"{""str"": 1}","{""string"": 1}",1,False,24.0,24.0,"해설서, 사례/20220406_온라인_식품·건강기능식품·의약외품·화장품_분야_부당광..."
59,I0030_20261122_202624.json,CSTDY_MTHD,46501,46501,3224,43277,0.930668,3224,"{""str"": 3224}","{""string"": 3224}",893,False,1.0,205.0,01_건강기능식품/원본 데이터/제품정보-20261123T031715Z-1-001/I...
249,식품관련법 위반사례_식품안전나라.json,DSPS_ENDDT,5060,5060,371,4689,0.926680,371,"{""str"": 371}","{""date_string"": 371}",115,False,8.0,8.0,"해설서, 사례/식품관련법 위반사례_식품안전나라.json"
131,I1250_261123.json,QLITY_MNTNC_TMLMT_DAYCNT,1087955,1087955,183920,904035,0.830949,10947,"{""str"": 10947}","{""string"": 10924, ""date_string"": 1, ""integer_s...",437,False,1.0,69.0,02_일반식품/원본 데이터/I1250/I1250_261123.json


,file_name,column_name,sampled_non_missing_count,valid_date_count,invalid_date_count,valid_date_rate,detected_formats,relative_path
0,건강기능식품_기능성_원료인정현황.json,PRMS_DT,779,779,0,1.0,"{""YYYYMMDD"": 779}",01_건강기능식품/원본 데이터/기능성원료-20261123T031711Z-1-001/...
1,I0030_20261122_202624.json,LAST_UPDT_DTM,46501,46501,0,1.0,"{""YYYYMMDD"": 46501}",01_건강기능식품/원본 데이터/제품정보-20261123T031715Z-1-001/I...
2,I0030_20261122_202624.json,PRMS_DT,46501,46501,0,1.0,"{""YYYYMMDD"": 46501}",01_건강기능식품/원본 데이터/제품정보-20261123T031715Z-1-001/I...
3,C002_261123.json,CHNG_DT,100000,100000,0,1.0,"{""YYYYMMDD"": 100000}",02_일반식품/원본 데이터/C002/C002_261123.json
4,C002_261123.json,PRMS_DT,100000,100000,0,1.0,"{""YYYYMMDD"": 100000}",02_일반식품/원본 데이터/C002/C002_261123.json
5,I1250_261123.json,LAST_UPDT_DTM,100000,100000,0,1.0,"{""YYYYMMDD"": 100000}",02_일반식품/원본 데이터/I1250/I1250_261123.json
6,I1250_261123.json,PRMS_DT,100000,100000,0,1.0,"{""YYYYMMDD"": 100000}",02_일반식품/원본 데이터/I1250/I1250_261123.json
7,food_raw_material_metadata.json,collected_at,1,1,0,1.0,"{""ISO_DATETIME"": 1}",02_일반식품/원본 데이터/식품원료/food_raw_material_metadata...
8,식품관련법 위반사례_식품안전나라.json,DSPS_BGNDT,4846,4846,0,1.0,"{""YYYYMMDD"": 4846}","해설서, 사례/식품관련법 위반사례_식품안전나라.json"
9,식품관련법 위반사례_식품안전나라.json,DSPS_ENDDT,371,371,0,1.0,"{""YYYYMMDD"": 371}","해설서, 사례/식품관련법 위반사례_식품안전나라.json"


### 결과 해석

- \`missing_rate\`: 전체 행 중 결측인 비율입니다.
- \`physical_types\`: JSON에 실제 저장된 Python 타입입니다. CSV를 \`DictReader\`로 파싱한 데이터는 대부분 문자열로 나타납니다.
- \`inferred_types\`: 문자열 내용을 정수·실수·불리언·날짜로 추론한 결과입니다.
- \`valid_date_rate\`: 날짜 후보 컬럼에서 실제 달력상 유효한 값의 비율입니다.
- \`distinct_truncated=True\`: 고유값이 설정한 한도를 넘어 정확한 고유값 수 대신 고유값이 많다는 사실만 기록한 경우입니다.

## 7. 키 중복과 I1250–C002 조인 가능성

키별 결측, 고유값, 중복 키, 키 하나당 최대 행 수를 확인합니다.

I1250과 C002는 \`PRDLST_REPORT_NO\`로 외부 조인한다고 가정하고 다음을 계산합니다.

- 양쪽 키 교집합과 한쪽에만 존재하는 키
- 각 파일의 중복 키
- 실제 내부 조인 시 예상 행 수와 중복으로 인한 행 증가
- 같은 제품번호의 \`LCNS_NO\` 일치 여부

In [7]:
left_name = "I1250_261123.json"
right_name = "C002_261123.json"

left_counter = key_counters_by_file.get(left_name, {}).get("PRDLST_REPORT_NO")
right_counter = key_counters_by_file.get(right_name, {}).get("PRDLST_REPORT_NO")

if left_counter is not None and right_counter is not None:
    join_result = profile_join(
        left_counter,
        right_counter,
        license_maps_by_file.get(left_name),
        license_maps_by_file.get(right_name),
    )
    join_result.update({"left_file": left_name, "right_file": right_name, "status": "analyzed"})
else:
    join_result = {
        "left_file": left_name,
        "right_file": right_name,
        "status": "required_file_or_key_missing",
    }

join_profile_df = pd.DataFrame([join_result])

display(key_profile_df)
display(duplicate_key_samples_df.head(100))
display(join_profile_df.T)

,file_name,key_column,row_count,non_missing_key_count,missing_key_count,unique_key_count,duplicate_key_count,duplicate_row_count,max_rows_per_key,is_unique,relative_path
0,건강기능식품_개별인정형_정보.json,HF_FNCLTY_MTRAL_RCOGN_NO,457,457,0,455,2,2,2,False,01_건강기능식품/원본 데이터/개별인정-20261123T031709Z-1-001/건...
1,건강기능식품_기능성_원료인정현황.json,HF_FNCLTY_MTRAL_RCOGN_NO,779,778,1,774,4,4,2,False,01_건강기능식품/원본 데이터/기능성원료-20261123T031711Z-1-001/...
2,I0030_20261122_202624.json,PRDLST_REPORT_NO,46501,46501,0,46501,0,0,1,True,01_건강기능식품/원본 데이터/제품정보-20261123T031715Z-1-001/I...
3,I0030_20261122_202624.json,LCNS_NO,46501,46501,0,520,444,45981,1889,False,01_건강기능식품/원본 데이터/제품정보-20261123T031715Z-1-001/I...
4,C002_261123.json,PRDLST_REPORT_NO,1087964,1087964,0,1087964,0,0,1,True,02_일반식품/원본 데이터/C002/C002_261123.json
5,C002_261123.json,LCNS_NO,1087964,1087964,0,31306,27305,1056658,13522,False,02_일반식품/원본 데이터/C002/C002_261123.json
6,I1250_261123.json,PRDLST_REPORT_NO,1087955,1087955,0,1087955,0,0,1,True,02_일반식품/원본 데이터/I1250/I1250_261123.json
7,I1250_261123.json,LCNS_NO,1087955,1087955,0,31305,27304,1056650,13522,False,02_일반식품/원본 데이터/I1250/I1250_261123.json
8,식품관련법 위반사례_식품안전나라.json,LCNS_NO,5060,5060,0,5031,29,29,2,False,"해설서, 사례/식품관련법 위반사례_식품안전나라.json"
9,식품관련법 위반사례_식품안전나라.json,DSPSDTLS_SEQ,5060,5060,0,5059,1,1,2,False,"해설서, 사례/식품관련법 위반사례_식품안전나라.json"


,file_name,key_column,key_value,row_count_for_key,relative_path
0,건강기능식품_개별인정형_정보.json,HF_FNCLTY_MTRAL_RCOGN_NO,제2013-5호,2,01_건강기능식품/원본 데이터/개별인정-20261123T031709Z-1-001/건...
1,건강기능식품_개별인정형_정보.json,HF_FNCLTY_MTRAL_RCOGN_NO,제2021-5호,2,01_건강기능식품/원본 데이터/개별인정-20261123T031709Z-1-001/건...
2,건강기능식품_기능성_원료인정현황.json,HF_FNCLTY_MTRAL_RCOGN_NO,2007-10,2,01_건강기능식품/원본 데이터/기능성원료-20261123T031711Z-1-001/...
3,건강기능식품_기능성_원료인정현황.json,HF_FNCLTY_MTRAL_RCOGN_NO,2007-11,2,01_건강기능식품/원본 데이터/기능성원료-20261123T031711Z-1-001/...
4,건강기능식품_기능성_원료인정현황.json,HF_FNCLTY_MTRAL_RCOGN_NO,2007-20,2,01_건강기능식품/원본 데이터/기능성원료-20261123T031711Z-1-001/...
...,...,...,...,...,...
95,I0030_20261122_202624.json,LCNS_NO,20130017122,93,01_건강기능식품/원본 데이터/제품정보-20261123T031715Z-1-001/I...
96,I0030_20261122_202624.json,LCNS_NO,20240020233,90,01_건강기능식품/원본 데이터/제품정보-20261123T031715Z-1-001/I...
97,I0030_20261122_202624.json,LCNS_NO,20040019002,89,01_건강기능식품/원본 데이터/제품정보-20261123T031715Z-1-001/I...
98,I0030_20261122_202624.json,LCNS_NO,20040020036,87,01_건강기능식품/원본 데이터/제품정보-20261123T031715Z-1-001/I...


,0
join_key,PRDLST_REPORT_NO
left_unique_key_count,1087955
right_unique_key_count,1087964
both_key_count,1087872
left_only_key_count,83
right_only_key_count,92
left_overlap_rate,0.999924
right_overlap_rate,0.999915
left_duplicate_key_count,0
right_duplicate_key_count,0


## 8. PDF 파싱 품질

\`document + pages\` 구조 파일에 대해 빈 페이지와 텍스트 추출량을 검사합니다.

여기서 텍스트 추출률은 **전체 페이지 중 공백이 아닌 텍스트가 있는 페이지의 비율**입니다. PDF 원본의 모든 글자가 정확히 추출됐다는 의미는 아닙니다.

다음 중 하나면 \`needs_reparse_review=True\`로 표시합니다.

- 빈 페이지 비율이 20% 이상
- 페이지당 평균 추출 문자가 100자 미만
- 대체문자 \`�\`가 한 개 이상 존재

In [8]:
pdf_rows = []
page_files = file_inventory_df[
    file_inventory_df["included"] & file_inventory_df["json_shape"].eq("pages")
]
for _, row in page_files.iterrows():
    path = DATA_ROOT / row["relative_path"]
    try:
        result = profile_pdf_pages(path)
        result["relative_path"] = row["relative_path"]
        result["error"] = ""
    except Exception as error:
        result = {
            "file_name": path.name,
            "relative_path": row["relative_path"],
            "error": f"{type(error).__name__}: {error}",
        }
    pdf_rows.append(result)

pdf_extraction_profile_df = pd.DataFrame(pdf_rows)
display(
    pdf_extraction_profile_df.sort_values(
        ["needs_reparse_review", "nonempty_page_rate"],
        ascending=[False, True],
    )
)

,file_name,declared_page_count,actual_page_count,page_count_matches,empty_page_count,nonempty_page_count,nonempty_page_rate,total_extracted_characters,average_characters_per_page,median_characters_per_page,replacement_character_count,needs_reparse_review,relative_path,error
26,20220406_온라인_식품·건강기능식품·의약외품·화장품_분야_부당광고_사례집-웹게...,84,84,True,83,1,0.011905,24,0.285714,0.0,0,True,"해설서, 사례/20220406_온라인_식품·건강기능식품·의약외품·화장품_분야_부당광...",
23,☆+「식품등의+부당한+표시+또는+광고의+내용+기준」+고시+전문(제2024-23호，+...,13,13,True,1,12,0.923077,228,17.538462,7.0,0,True,law/☆+「식품등의+부당한+표시+또는+광고의+내용+기준」+고시+전문(제2024-2...,
27,230216_식품·의약품등의_온라인_자율관리_가이드라인.json,64,64,True,5,59,0.921875,52365,818.203125,826.0,0,False,"해설서, 사례/230216_식품·의약품등의_온라인_자율관리_가이드라인.json",
25,20210225_식품등_부당한_광고_판단기준_가이드라인(판단안내서)_수정.json,133,133,True,10,123,0.924812,150204,1129.353383,1024.0,0,False,"해설서, 사례/20210225_식품등_부당한_광고_판단기준_가이드라인(판단안내서)_...",
28,합본_식품_등_온라인_부당광고_사례집.json,78,78,True,3,75,0.961538,27556,353.282051,163.0,0,False,"해설서, 사례/합본_식품_등_온라인_부당광고_사례집.json",
5,건강기능식품의 기준 및 규격_(제4).json,558,558,True,5,553,0.991039,332578,596.017921,611.5,0,False,01_건강기능식품/법령_고시/현재/기준규격/제2026-43호/건강기능식품의 기준 및...,
0,식품등의 부당한 표시 또는 광고의 내용 기준_제2021-89호.json,4,4,True,0,4,1.000000,5156,1289.000000,1554.0,0,False,00_공통_표시광고-20261123T025100Z-1-001/과거/식품등의 부당한 ...,
1,"★++「식품등의+부당한+표시+또는+광고의+내용+기준」+고시+전문(제2025-79호,...",27,27,True,0,27,1.000000,16380,606.666667,676.0,0,False,00_공통_표시광고-20261123T025100Z-1-001/현재/★++「식품등의+...,
2,건강기능식품의 표시기준(식품의약품안전처고시)(제2026-66호)(20280101)....,10,10,True,0,10,1.000000,14452,1445.200000,1561.0,0,False,01_건강기능식품/법령_고시/원본 데이터/시행예정/2028/건강기능식품의 표시기준(...,
3,건강기능식품의 기준 및 규격(식품의약품안전처고시)(제2026-43호)(2026061...,1,1,True,0,1,1.000000,185,185.000000,185.0,0,False,01_건강기능식품/법령_고시/현재/기준규격/제2026-43호/건강기능식품의 기준 및...,


## 9. 파일명과 내부 메타데이터 일치

다음 정보를 비교합니다.

- 현재 JSON 파일명과 \`source_pdf/source_csv\`의 원본 파일명
- 현재 파일명과 내부 \`document.title\`
- 국가법령정보 구조에서는 \`법령명_한글\`
- 제2025-79호와 C002/I1250은 알려진 이슈 ID를 함께 표시

제목 유사도는 0~1이며 낮을수록 파일명과 내부 제목이 다릅니다. 파일명이 코드명인 C002/I1250처럼 자동 판정만으로 확정하기 어려운 경우에는 알려진 이슈와 함께 사람이 검토해야 합니다.

In [9]:
metadata_rows = []
for _, row in file_inventory_df[file_inventory_df["included"]].iterrows():
    path = DATA_ROOT / row["relative_path"]
    try:
        result = compare_metadata(path, row["json_shape"])
        result["relative_path"] = row["relative_path"]
        result["error"] = ""
    except Exception as error:
        result = {
            "file_name": path.name,
            "relative_path": row["relative_path"],
            "metadata_status": "analysis_error",
            "error": f"{type(error).__name__}: {error}",
        }
    metadata_rows.append(result)

metadata_consistency_df = pd.DataFrame(metadata_rows)
display(
    metadata_consistency_df.sort_values(
        ["metadata_status", "title_filename_similarity"],
        ascending=[True, True],
    )
)

,file_name,internal_title,source_name,source_name_matches_file,title_filename_similarity,metadata_status,known_issue_id,notice_number,published_date,relative_path,error
17,C002_261123.json,건강기능식품 개별인정형 정보,C002_261123.csv,True,0.000000,known_issue,KNOWN-005,None,None,02_일반식품/원본 데이터/C002/C002_261123.json,
18,I1250_261123.json,건강기능식품 개별인정형 정보,I1250_261123.csv,True,0.000000,known_issue,KNOWN-005,None,None,02_일반식품/원본 데이터/I1250/I1250_261123.json,
1,"★++「식품등의+부당한+표시+또는+광고의+내용+기준」+고시+전문(제2025-79호,...",건강기능식품의 표시기준 고시전문,"★건강기능식품의 표시기준 고시전문(고시 제2025-61호, 2025.8.29.).pdf",False,0.423077,known_issue,KNOWN-003,고시 제2025-61호,2025-08-29,00_공통_표시광고-20261123T025100Z-1-001/현재/★++「식품등의+...,
14,식품등의 표시기준(식품의약품안전처고시)(제2026-37호)(20280101).json,건강기능식품의 표시기준 고시전문,식품등의 표시기준(식품의약품안전처고시)(제2026-37호)(20280101).pdf,True,0.367347,match_or_review,,고시 제2025-61호,2025-08-29,02_일반식품/법령_고시/시행예정/2028/표시기준/식품등의 표시기준(식품의약품안전...,
3,건강기능식품의 기준 및 규격(식품의약품안전처고시)(제2026-43호)(2026061...,건강기능식품의 표시기준 고시전문,건강기능식품의 기준 및 규격(식품의약품안전처고시)(제2026-43호)(2026061...,True,0.415094,match_or_review,,고시 제2025-61호,2025-08-29,01_건강기능식품/법령_고시/현재/기준규격/제2026-43호/건강기능식품의 기준 및...,
32,☆+「식품등의+부당한+표시+또는+광고의+내용+기준」+고시+전문(제2024-23호，+...,건강기능식품의 표시기준 고시전문,☆+「식품등의+부당한+표시+또는+광고의+내용+기준」+고시+전문(제2024-23호，+...,True,0.423077,match_or_review,,고시 제2025-61호,2025-08-29,law/☆+「식품등의+부당한+표시+또는+광고의+내용+기준」+고시+전문(제2024-2...,
16,식품등의 표시기준_20260517기준.json,건강기능식품의 표시기준 고시전문,식품등의 표시기준_20260517기준.pdf,True,0.424242,match_or_review,,고시 제2025-61호,2025-08-29,02_일반식품/법령_고시/현재/표시기준/식품등의 표시기준_20260517기준.json,
36,"건강기능식품의+기준+및+규격+일부개정고시(제2026-43호,+2026.6.11.)....",건강기능식품의 표시기준 고시전문,"건강기능식품의+기준+및+규격+일부개정고시(제2026-43호,+2026.6.11.).pdf",True,0.458333,match_or_review,,고시 제2025-61호,2025-08-29,"law/건강기능식품의+기준+및+규격+일부개정고시(제2026-43호,+2026.6.1...",
2,건강기능식품의 표시기준(식품의약품안전처고시)(제2026-66호)(20280101)....,건강기능식품의 표시기준 고시전문,건강기능식품의 표시기준(식품의약품안전처고시)(제2026-66호)(20280101).pdf,True,0.500000,match_or_review,,고시 제2025-61호,2025-08-29,01_건강기능식품/법령_고시/원본 데이터/시행예정/2028/건강기능식품의 표시기준(...,
10,건강기능식품_영양DB.json,건강기능식품 개별인정형 정보,건강기능식품_영양DB.csv,True,0.521739,match_or_review,,None,None,01_건강기능식품/원본 데이터/영양DB-20261123T031713Z-1-001/건...,


## 10. 보고서 저장

CSV는 Excel에서도 한글이 깨지지 않도록 UTF-8 BOM으로 저장합니다. JSON은 한글을 그대로 보존합니다.

노트북을 다시 실행하면 동일한 이름의 이전 EDA 보고서를 갱신하지만, \`data_parsing\` 파일에는 영향을 주지 않습니다.

In [10]:
reports = {
    "file_inventory.csv": file_inventory_df,
    "dataset_profile.csv": dataset_profile_df,
    "column_profile.csv": column_profile_df,
    "date_profile.csv": date_profile_df,
    "key_profile.csv": key_profile_df,
    "duplicate_key_samples.csv": duplicate_key_samples_df,
    "join_profile.csv": join_profile_df,
    "pdf_extraction_profile.csv": pdf_extraction_profile_df,
    "metadata_consistency.csv": metadata_consistency_df,
}

required_columns = {
    "file_inventory.csv": {"relative_path", "included", "json_shape"},
    "dataset_profile.csv": {"relative_path", "actual_record_or_page_count", "error"},
    "column_profile.csv": {"relative_path", "column_name", "missing_rate"},
    "date_profile.csv": {"relative_path", "column_name", "detected_formats"},
    "key_profile.csv": {"relative_path", "key_column", "duplicate_key_count"},
    "join_profile.csv": {"left_file", "right_file", "status"},
    "pdf_extraction_profile.csv": {"relative_path", "empty_page_count", "nonempty_page_rate"},
    "metadata_consistency.csv": {"relative_path", "metadata_status"},
}

for file_name, expected_columns in required_columns.items():
    actual_columns = set(reports[file_name].columns)
    missing_columns = expected_columns - actual_columns
    assert not missing_columns, f"{file_name} 필수 컬럼 누락: {sorted(missing_columns)}"

for file_name, frame in reports.items():
    frame.to_csv(OUTPUT_DIR / file_name, index=False, encoding="utf-8-sig")

with (OUTPUT_DIR / "known_issues.json").open("w", encoding="utf-8") as stream:
    json.dump(KNOWN_ISSUES, stream, ensure_ascii=False, indent=2)

summary = {
    "generated_at": datetime.now().astimezone().isoformat(),
    "data_root": str(DATA_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "full_scan": FULL_SCAN,
    "profile_sample_rows": PROFILE_SAMPLE_ROWS,
    "discovered_file_count": int(len(file_inventory_df)),
    "included_file_count": int(file_inventory_df["included"].sum()),
    "excluded_file_count": int((~file_inventory_df["included"]).sum()),
    "profile_error_count": int(dataset_profile_df["error"].fillna("").ne("").sum()),
    "exact_duplicate_file_count": int(file_inventory_df["exact_duplicate_file"].sum()),
    "pdf_reparse_review_count": int(
        pdf_extraction_profile_df.get("needs_reparse_review", pd.Series(dtype=bool))
        .fillna(False)
        .sum()
    ),
    "metadata_mismatch_or_known_issue_count": int(
        metadata_consistency_df["metadata_status"].isin(["mismatch", "known_issue"]).sum()
    ),
    "known_issue_count": len(KNOWN_ISSUES),
    "reports": sorted([*reports, "known_issues.json", "eda_summary.json"]),
}
with (OUTPUT_DIR / "eda_summary.json").open("w", encoding="utf-8") as stream:
    json.dump(summary, stream, ensure_ascii=False, indent=2)

print("저장 완료")
for file_name in summary["reports"]:
    print(f"- {OUTPUT_DIR / file_name}")

저장 완료
- C:\Users\Playdata\Desktop\mle-01-final-team3\data\eda_outputs\column_profile.csv
- C:\Users\Playdata\Desktop\mle-01-final-team3\data\eda_outputs\dataset_profile.csv
- C:\Users\Playdata\Desktop\mle-01-final-team3\data\eda_outputs\date_profile.csv
- C:\Users\Playdata\Desktop\mle-01-final-team3\data\eda_outputs\duplicate_key_samples.csv
- C:\Users\Playdata\Desktop\mle-01-final-team3\data\eda_outputs\eda_summary.json
- C:\Users\Playdata\Desktop\mle-01-final-team3\data\eda_outputs\file_inventory.csv
- C:\Users\Playdata\Desktop\mle-01-final-team3\data\eda_outputs\join_profile.csv
- C:\Users\Playdata\Desktop\mle-01-final-team3\data\eda_outputs\key_profile.csv
- C:\Users\Playdata\Desktop\mle-01-final-team3\data\eda_outputs\known_issues.json
- C:\Users\Playdata\Desktop\mle-01-final-team3\data\eda_outputs\metadata_consistency.csv
- C:\Users\Playdata\Desktop\mle-01-final-team3\data\eda_outputs\pdf_extraction_profile.csv


## 11. 구조화 설계 전 판단 체크리스트

보고서를 확인한 뒤 다음을 결정합니다.

- 기본키 후보가 실제로 유일한가?
- 중복은 완전 중복, 버전 이력, 충돌 중 어느 유형인가?
- I1250–C002 조인은 1:1인가, 1:N인가?
- 날짜 컬럼을 실제 날짜 타입으로 변환할 수 있는가?
- 결측 표현을 어떤 값으로 통일할 것인가?
- 재파싱 또는 OCR이 필요한 PDF는 무엇인가?
- 파일명과 내부 제목 중 어느 정보를 신뢰할 것인가?
- 현재·과거·시행예정 법령의 유효기간을 어떤 필드로 관리할 것인가?

알려진 문제와 새로 발견된 문제를 구분한 다음 표준 스키마를 확정하는 것이 안전합니다.